# PatrolIQ — Data Cleaning and Feature Engineering

This notebook prepares the Chicago crime data for analysis by validating data quality, selecting the 500,000 most recent cleaned records, and engineering temporal, geographic, categorical, and severity features.


In [1]:
import pandas as pd

# Load raw Chicago crime dataset
DATA_PATH = "../data/raw/chicago_crime_raw.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print("\nColumn names:")
print(df.columns.tolist())

Dataset shape: (847276, 22)
Rows: 847,276
Columns: 22

Column names:
['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


In [2]:
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

DATA TYPES
ID                        int64
Case Number              object
Date                     object
Block                    object
IUCR                     object
Primary Type             object
Description              object
Location Description     object
Arrest                     bool
Domestic                   bool
Beat                      int64
District                  int64
Ward                    float64
Community Area          float64
FBI Code                 object
X Coordinate            float64
Y Coordinate            float64
Year                      int64
Updated On               object
Latitude                float64
Longitude               float64
Location                 object
dtype: object

MISSING VALUES
X Coordinate            5231
Y Coordinate            5231
Latitude                5231
Longitude               5231
Location                5231
Location Description    4212
Community Area            36
Ward                       4
dtype: int64

DUPLICATE

In [3]:
raw_check = pd.read_csv(
    "../data/raw/chicago_crime_raw.csv",
    usecols=["Date"],
    nrows=10
)

print(raw_check["Date"].tolist())

['05/29/2026 08:30:00 PM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM', '05/26/2026 12:00:00 AM']


In [4]:
# ============================================================
# DATA CLEANING
# ============================================================

# Convert Date to datetime using the verified raw format
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

# Fill missing location descriptions instead of dropping records
df["Location Description"] = (
    df["Location Description"]
    .fillna("UNKNOWN")
)

# Remove records missing essential temporal,
# geographic, or administrative information
df = df.dropna(
    subset=[
        "Date",
        "Latitude",
        "Longitude",
        "X Coordinate",
        "Y Coordinate",
        "Ward",
        "Community Area"
    ]
).copy()

# Reset index after removing invalid records
df.reset_index(drop=True, inplace=True)

print(f"Shape after cleaning: {df.shape}")
print(f"Rows removed: {847276 - len(df):,}")

Shape after cleaning: (842007, 22)
Rows removed: 5,269


In [5]:
# ============================================================
# POST-CLEANING DATA QUALITY CHECK
# ============================================================

print("Shape after cleaning:", df.shape)

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nColumns with missing values:")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0])

print("\nDuplicate rows:", df.duplicated().sum())

Shape after cleaning: (842007, 22)

Total missing values: 0

Columns with missing values:
Series([], dtype: int64)

Duplicate rows: 0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 842007 entries, 0 to 842006
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   ID                    842007 non-null  int64         
 1   Case Number           842007 non-null  object        
 2   Date                  842007 non-null  datetime64[ns]
 3   Block                 842007 non-null  object        
 4   IUCR                  842007 non-null  object        
 5   Primary Type          842007 non-null  object        
 6   Description           842007 non-null  object        
 7   Location Description  842007 non-null  object        
 8   Arrest                842007 non-null  bool          
 9   Domestic              842007 non-null  bool          
 10  Beat                  842007 non-null  int64         
 11  District              842007 non-null  int64         
 12  Ward                  842007 non-null  float64       
 13 

In [7]:
# Select the 500,000 most recent cleaned crime records
df = (
    df.sort_values("Date", ascending=False)
      .head(500_000)
      .reset_index(drop=True)
)

# Validate the analysis dataset
print("Analysis dataset shape:", df.shape)
print("Latest crime date:", df["Date"].max())
print("Oldest crime date:", df["Date"].min())
print("Duplicate rows:", df.duplicated().sum())
print("Total missing values:", df.isnull().sum().sum())

Analysis dataset shape: (500000, 22)
Latest crime date: 2026-05-29 20:30:00
Oldest crime date: 2024-05-01 10:20:00
Duplicate rows: 0
Total missing values: 0


## Feature Engineering


In [8]:
# Hour
df['Hour'] = df['Date'].dt.hour

# Day of Week
df['Day_of_Week'] = df['Date'].dt.day_name()

# Month
df['Month'] = df['Date'].dt.month

# Weekend Flag
df['Is_Weekend'] = df['Day_of_Week'].isin(
    ['Saturday', 'Sunday']
).astype(int)

# Season
season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}

df['Season'] = df['Month'].map(season_map)

df[['Date','Hour','Day_of_Week','Month','Is_Weekend','Season']].head()

,Date,Hour,Day_of_Week,Month,Is_Weekend,Season
0,2026-05-29 20:30:00,20,Friday,5,0,Spring
1,2026-05-26 00:00:00,0,Tuesday,5,0,Spring
2,2026-05-26 00:00:00,0,Tuesday,5,0,Spring
3,2026-05-26 00:00:00,0,Tuesday,5,0,Spring
4,2026-05-26 00:00:00,0,Tuesday,5,0,Spring


## Crime Severity Score


In [9]:
severity_map = {
    'HOMICIDE': 10,
    'KIDNAPPING': 9,
    'CRIMINAL SEXUAL ASSAULT': 8,
    'ROBBERY': 7,
    'ASSAULT': 6,
    'BURGLARY': 5,
    'THEFT': 4
}

df['Crime_Severity_Score'] = (
    df['Primary Type']
    .map(severity_map)
    .fillna(3)
    .astype(int)
)

In [10]:
df[['Primary Type','Crime_Severity_Score']].head(10)

,Primary Type,Crime_Severity_Score
0,THEFT,4
1,CRIMINAL SEXUAL ASSAULT,8
2,CRIMINAL DAMAGE,3
3,INTERFERENCE WITH PUBLIC OFFICER,3
4,THEFT,4
5,THEFT,4
6,THEFT,4
7,DECEPTIVE PRACTICE,3
8,DECEPTIVE PRACTICE,3
9,MOTOR VEHICLE THEFT,3


## Additional Feature Engineering


In [12]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ============================================================
# GEOGRAPHIC AND CATEGORICAL FEATURE ENGINEERING
# ============================================================

# 1. Geographic Coordinate Binning
# Groups nearby incidents into approximate geographic grid cells
df["Latitude_Bin"] = df["Latitude"].round(2)
df["Longitude_Bin"] = df["Longitude"].round(2)

df["Geo_Bin"] = (
    df["Latitude_Bin"].astype(str)
    + "_"
    + df["Longitude_Bin"].astype(str)
)


# 2. Administrative Geographic Group
# District is an existing police administrative boundary.
# Actual district-level clustering will be evaluated later.
df["District_Group"] = df["District"].astype(str)


# 3. Categorical Encoding
crime_encoder = LabelEncoder()
location_encoder = LabelEncoder()

df["Primary_Type_Encoded"] = crime_encoder.fit_transform(
    df["Primary Type"].astype(str)
)

df["Location_Description_Encoded"] = location_encoder.fit_transform(
    df["Location Description"].astype(str)
)


# 4. Normalize Geographic Coordinates
scaler = StandardScaler()

df[["Latitude_Scaled", "Longitude_Scaled"]] = scaler.fit_transform(
    df[["Latitude", "Longitude"]]
)


# 5. Verify Engineered Features
new_features = [
    "Latitude_Bin",
    "Longitude_Bin",
    "Geo_Bin",
    "District_Group",
    "Primary_Type_Encoded",
    "Location_Description_Encoded",
    "Latitude_Scaled",
    "Longitude_Scaled"
]

df[new_features].head()

,Latitude_Bin,Longitude_Bin,Geo_Bin,District_Group,Primary_Type_Encoded,Location_Description_Encoded,Latitude_Scaled,Longitude_Scaled
0,41.84,-87.66,41.84_-87.66,9,29,16,-0.090237,0.196078
1,41.87,-87.73,41.87_-87.73,10,6,119,0.221197,-1.032442
2,41.82,-87.60,41.82_-87.6,2,5,103,-0.297744,1.130786
3,41.77,-87.62,41.77_-87.62,3,12,119,-0.895952,0.873661
4,42.00,-87.80,42.0_-87.8,16,29,119,1.736145,-2.189646


In [13]:
# Validation before saving
print("Prepared dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Total columns:", df.shape[1])
print("Latest record:", df["Date"].max())
print("Oldest record:", df["Date"].min())

# Save featured dataset
OUTPUT_PATH = "../data/processed/chicago_crime_featured.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("\nDataset saved successfully:")
print(OUTPUT_PATH)

Prepared dataset shape: (500000, 36)
Total missing values: 0
Duplicate rows: 0
Total columns: 36
Latest record: 2026-05-29 20:30:00
Oldest record: 2024-05-01 10:20:00

Dataset saved successfully:
../data/processed/chicago_crime_featured.csv


In [14]:
# ============================================================
# PREPROCESSING & FEATURE ENGINEERING VALIDATION
# ============================================================

required_original_columns = [
    "ID",
    "Case Number",
    "Date",
    "Block",
    "IUCR",
    "Primary Type",
    "Description",
    "Location Description",
    "Arrest",
    "Domestic",
    "Beat",
    "District",
    "Ward",
    "Community Area",
    "FBI Code",
    "X Coordinate",
    "Y Coordinate",
    "Year",
    "Updated On",
    "Latitude",
    "Longitude",
    "Location"
]

required_engineered_columns = [
    "Hour",
    "Day_of_Week",
    "Month",
    "Is_Weekend",
    "Season",
    "Crime_Severity_Score",
    "Latitude_Bin",
    "Longitude_Bin",
    "Geo_Bin",
    "District_Group",
    "Primary_Type_Encoded",
    "Location_Description_Encoded",
    "Latitude_Scaled",
    "Longitude_Scaled"
]

print("=" * 65)
print("DATA PREPARATION VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Dataset size
# ------------------------------------------------------------

print("\n1. DATASET SIZE")
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])

if len(df) == 500_000:
    print("✓ Required 500,000 analysis records available")
else:
    print("✗ Dataset does not contain exactly 500,000 records")


# ------------------------------------------------------------
# 2. Original columns
# ------------------------------------------------------------

missing_original = [
    col for col in required_original_columns
    if col not in df.columns
]

print("\n2. ORIGINAL FEATURES")

if not missing_original:
    print("✓ All 22 original dataset columns are present")
else:
    print("✗ Missing original columns:")
    print(missing_original)


# ------------------------------------------------------------
# 3. Engineered features
# ------------------------------------------------------------

missing_engineered = [
    col for col in required_engineered_columns
    if col not in df.columns
]

print("\n3. ENGINEERED FEATURES")

if not missing_engineered:
    print("✓ All required engineered features are present")
else:
    print("✗ Missing engineered features:")
    print(missing_engineered)


# ------------------------------------------------------------
# 4. Missing values and duplicates
# ------------------------------------------------------------

print("\n4. DATA QUALITY")

total_missing = df.isnull().sum().sum()
duplicates = df.duplicated().sum()

print("Total missing values:", total_missing)
print("Duplicate rows:", duplicates)

if total_missing == 0:
    print("✓ No missing values")
else:
    print("✗ Missing values still exist")

if duplicates == 0:
    print("✓ No duplicate rows")
else:
    print("✗ Duplicate rows detected")


# ------------------------------------------------------------
# 5. Date coverage
# ------------------------------------------------------------

print("\n5. DATE COVERAGE")

print("Oldest record:", df["Date"].min())
print("Latest record:", df["Date"].max())


# ------------------------------------------------------------
# 6. Temporal feature validation
# ------------------------------------------------------------

print("\n6. TEMPORAL FEATURES")

print(
    "Hour range:",
    df["Hour"].min(),
    "to",
    df["Hour"].max()
)

print(
    "Months represented:",
    sorted(df["Month"].unique())
)

print(
    "Days represented:",
    df["Day_of_Week"].nunique()
)

print(
    "Seasons:",
    sorted(df["Season"].unique())
)

print(
    "Weekend values:",
    sorted(df["Is_Weekend"].unique())
)


# ------------------------------------------------------------
# 7. Geographic validation
# ------------------------------------------------------------

print("\n7. GEOGRAPHIC FEATURES")

print(
    "Latitude range:",
    df["Latitude"].min(),
    "to",
    df["Latitude"].max()
)

print(
    "Longitude range:",
    df["Longitude"].min(),
    "to",
    df["Longitude"].max()
)

print(
    "X Coordinate range:",
    df["X Coordinate"].min(),
    "to",
    df["X Coordinate"].max()
)

print(
    "Y Coordinate range:",
    df["Y Coordinate"].min(),
    "to",
    df["Y Coordinate"].max()
)

print(
    "Geographic bins:",
    df["Geo_Bin"].nunique()
)

print(
    "Districts:",
    df["District"].nunique()
)

print(
    "Community Areas:",
    df["Community Area"].nunique()
)

print(
    "Beats:",
    df["Beat"].nunique()
)


# ------------------------------------------------------------
# 8. Crime feature validation
# ------------------------------------------------------------

print("\n8. CRIME FEATURES")

print(
    "Crime types represented:",
    df["Primary Type"].nunique()
)

print(
    "Severity score range:",
    df["Crime_Severity_Score"].min(),
    "to",
    df["Crime_Severity_Score"].max()
)

print(
    "Encoded crime categories:",
    df["Primary_Type_Encoded"].nunique()
)

print(
    "Encoded location categories:",
    df["Location_Description_Encoded"].nunique()
)


# ------------------------------------------------------------
# 9. Validation status
# ------------------------------------------------------------

print("\n" + "=" * 65)

checks_passed = (
    len(df) == 500_000
    and total_missing == 0
    and duplicates == 0
    and len(missing_original) == 0
    and len(missing_engineered) == 0
)

if checks_passed:
    print("✓ DATA PREPARATION VALIDATION PASSED")
    print("Dataset is ready for EDA and clustering.")
else:
    print("✗ REVIEW REQUIRED BEFORE CONTINUING")

print("=" * 65)

DATA PREPARATION VALIDATION

1. DATASET SIZE
Rows: 500,000
Columns: 36
✓ Required 500,000 analysis records available

2. ORIGINAL FEATURES
✓ All 22 original dataset columns are present

3. ENGINEERED FEATURES
✓ All required engineered features are present

4. DATA QUALITY
Total missing values: 0
Duplicate rows: 0
✓ No missing values
✓ No duplicate rows

5. DATE COVERAGE
Oldest record: 2024-05-01 10:20:00
Latest record: 2026-05-29 20:30:00

6. TEMPORAL FEATURES
Hour range: 0 to 23
Months represented: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Days represented: 7
Seasons: ['Fall', 'Spring', 'Summer', 'Winter']
Weekend values: [np.int64(0), np.int64(1)]

7. GEOGRAPHIC FEATURES
Latitude range: 41.644589713 to 42.022558955
Longitude range: -87.934567226 to -87.524529378
X Coordinate range: 1092647.0 to 1205119.0
Y Coordinate range: 1813897.0 to 1951517.0
Geographic bins: 724
